# Building a RAG System — Hands-on Workshop (OpenRouter)

In this notebook you will build a working **Retrieval-Augmented Generation (RAG)** chatbot step by step.

### What you will build
A chatbot that reads your company documents and answers questions accurately — without hallucinating.

### Pipeline
```
INDEXING (done once)
  Documents → Chunking → Embeddings → Vector Database

QUERYING (done per question)
  User Question → Embed → Search → Retrieve Context → LLM → Answer
```

---
**Requirements:** Make sure you ran `uv sync` and set your `OPENROUTER_API_KEY`.

## Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

# Load API key from .env file (if you have one)
load_dotenv()

# Or set it directly here (not recommended for production)
# os.environ["OPENROUTER_API_KEY"] = "sk-or-..."

# OpenRouter is OpenAI-compatible — just point the base_url to OpenRouter
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

print("Setup complete!")

---
## Step 1 — Load Documents

We load plain text files from the `sample_docs/` folder.  
In production this could be PDFs, web pages, databases, etc.

In [ ]:
def load_documents(folder: str) -> list[dict]:
    """Load all .txt files from a folder. Returns list of {filename, content}."""
    documents = []
    for path in Path(folder).glob("*.txt"):
        content = path.read_text(encoding="utf-8")
        documents.append({"filename": path.name, "content": content})
        print(f"Loaded: {path.name} ({len(content)} characters)")
    return documents


documents = load_documents("sample_docs")
print(f"\nTotal documents loaded: {len(documents)}")

---
## Step 2 — Chunk Documents

Documents are usually too long to fit into a single embedding or prompt.  
We split them into smaller **chunks** — each chunk is a unit that can be retrieved independently.

**Strategy:** Split by a fixed character count with overlap so that context is not lost at boundaries.

In [ ]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 100) -> list[str]:
    """
    Split text into overlapping chunks.
    chunk_size: max characters per chunk
    overlap: how many characters to repeat from the previous chunk
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk.strip())
        start += chunk_size - overlap  # move forward, keeping overlap
    return chunks


def chunk_documents(documents: list[dict]) -> list[dict]:
    """Chunk all documents and return a flat list of chunks with metadata."""
    all_chunks = []
    for doc in documents:
        chunks = chunk_text(doc["content"])
        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "id": f"{doc['filename']}_chunk_{i}",
                "text": chunk,
                "source": doc["filename"],
                "chunk_index": i,
            })
        print(f"{doc['filename']} → {len(chunks)} chunks")
    return all_chunks


chunks = chunk_documents(documents)
print(f"\nTotal chunks: {len(chunks)}")

# Preview first chunk
print("\n--- First chunk preview ---")
print(chunks[0]["text"][:300])

---
## Step 3 — Generate Embeddings

An **embedding** converts text into a vector (list of numbers) that represents its **meaning**.  
Text with similar meaning will have vectors that point in a similar direction.

We use OpenAI's `text-embedding-3-small` model via OpenRouter (1536 dimensions).

In [ ]:
EMBEDDING_MODEL = "openai/text-embedding-3-small"

def get_embedding(text: str) -> list[float]:
    """Generate an embedding vector for a single text string."""
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text,
    )
    return response.data[0].embedding


def get_embeddings_batch(texts: list[str]) -> list[list[float]]:
    """Generate embeddings for a list of texts in one API call (more efficient)."""
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts,
    )
    # Results are returned in the same order as input
    return [item.embedding for item in response.data]


# Quick test
test_embedding = get_embedding("Hello, this is a test.")
print(f"Embedding dimension: {len(test_embedding)}")
print(f"First 5 values: {test_embedding[:5]}")

---
## Step 4 — Store in Vector Database

We use **ChromaDB** — a local vector database that runs in memory.  
It stores each chunk along with its embedding, making it searchable.

This step is called **indexing**.

In [ ]:
# Initialize ChromaDB (in-memory for this workshop)
chroma_client = chromadb.Client()

# Create a collection (like a table in a regular database)
collection = chroma_client.create_collection(
    name="rag_workshop",
    metadata={"hnsw:space": "cosine"},  # use cosine similarity
)

print("Vector database initialized.")

In [ ]:
def index_chunks(chunks: list[dict], batch_size: int = 50):
    """Generate embeddings for all chunks and store them in ChromaDB."""
    total = len(chunks)
    for i in range(0, total, batch_size):
        batch = chunks[i : i + batch_size]
        texts = [c["text"] for c in batch]
        ids = [c["id"] for c in batch]
        metadatas = [{"source": c["source"], "chunk_index": c["chunk_index"]} for c in batch]

        embeddings = get_embeddings_batch(texts)

        collection.add(
            ids=ids,
            embeddings=embeddings,
            documents=texts,
            metadatas=metadatas,
        )
        print(f"Indexed {min(i + batch_size, total)}/{total} chunks...")

    print(f"\nDone! {total} chunks stored in the vector database.")


index_chunks(chunks)

---
## Step 5 — Implement Retrieval

When a user asks a question, we:
1. Convert the question to an embedding
2. Search the vector database for the most similar chunks
3. Return the top-K results as context

In [ ]:
def retrieve(query: str, top_k: int = 3) -> list[dict]:
    """
    Retrieve the top-K most relevant chunks for a given query.
    Returns list of {text, source, score}.
    """
    query_embedding = get_embedding(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )

    retrieved = []
    for text, metadata, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        retrieved.append({
            "text": text,
            "source": metadata["source"],
            "score": 1 - distance,  # convert distance → similarity score
        })

    return retrieved


# Test retrieval
test_results = retrieve("How many days of annual leave do I get?")

print(test_results)
print("Top retrieved chunks:\n")
for i, r in enumerate(test_results):
    print(f"[{i+1}] Source: {r['source']} | Score: {r['score']:.3f}")
    print(r["text"][:200])
    print()

---
## Step 6 — Generate Answer with LLM

We take the retrieved chunks, put them into a **prompt** as context, and ask the LLM to answer the question based on that context.

This is called **prompt augmentation** — the prompt is augmented with retrieved information.

In [ ]:
LLM_MODEL = "openai/gpt-4o-mini"

def generate_answer(query: str, context_chunks: list[dict]) -> str:
    """Generate an answer using the LLM, grounded in the retrieved context."""
    # Build the context string
    context = "\n\n---\n\n".join(
        f"Source: {c['source']}\n{c['text']}" for c in context_chunks
    )

    system_prompt = """You are a helpful assistant. Answer the user's question using ONLY the information provided in the context below.
If the answer is not in the context, say: "I don't have information about that in my documents."
Always cite the source document name when you answer."""

    user_prompt = f"""Context:
{context}

Question: {query}"""

    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
    )

    return response.choices[0].message.content


# Test with a single question
query = "How many days of annual leave do I get?"
context = retrieve(query)
answer = generate_answer(query, context)

print(f"Q: {query}")
print(f"\nA: {answer}")

---
## Putting It All Together — RAG Chatbot

Now we wrap everything into a single `ask()` function.

In [ ]:
def ask(question: str, top_k: int = 3, show_sources: bool = True) -> str:
    """Full RAG pipeline: retrieve relevant chunks, then generate an answer."""
    print(f"Question: {question}\n")

    # Retrieve
    context_chunks = retrieve(question, top_k=top_k)

    if show_sources:
        print("Retrieved sources:")
        for c in context_chunks:
            print(f"  - {c['source']} (score: {c['score']:.3f})")
        print()

    # Generate
    answer = generate_answer(question, context_chunks)
    print(f"Answer: {answer}")
    return answer


# Try different questions
ask("What is the refund policy?")

In [ ]:
ask("Can I work from home? How many days per week?")

In [ ]:
ask("How do I submit expense claims?")

In [ ]:
ask("What are the pricing plans for SmartAssist?")

In [ ]:
# Test: question that is NOT in the documents (should say it doesn't know)
ask("What is the capital of France?")

---
# Section 5 — Improving RAG Quality

Basic RAG works but has common problems. Here are practical improvements.

## Improvement 1 — Query Rewriting

Users often write vague or poorly phrased queries.  
We can use the LLM to rewrite the query before retrieval.

In [ ]:
def rewrite_query(original_query: str) -> str:
    """Use the LLM to rewrite the user's query for better retrieval."""
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {
                "role": "system",
                "content": "Rewrite the user's question to be clearer and more specific for a document search. Return only the rewritten question, nothing else.",
            },
            {"role": "user", "content": original_query},
        ],
        temperature=0,
    )
    return response.choices[0].message.content.strip()


def ask_with_rewrite(question: str, top_k: int = 3):
    """RAG pipeline with query rewriting."""
    rewritten = rewrite_query(question)
    print(f"Original  : {question}")
    print(f"Rewritten : {rewritten}\n")

    context_chunks = retrieve(rewritten, top_k=top_k)
    answer = generate_answer(question, context_chunks)  # answer original question
    print(f"Answer: {answer}")


ask_with_rewrite("leave days?")

## Improvement 2 — Reranking

Vector similarity is fast but not always the most accurate relevance signal.  
After retrieving more candidates, we can **rerank** them using the LLM to pick the best ones.

In [ ]:
def rerank(query: str, chunks: list[dict], top_k: int = 3) -> list[dict]:
    """
    Ask the LLM to score each chunk's relevance to the query.
    Returns the top_k most relevant chunks.
    """
    scored = []
    for chunk in chunks:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": "Rate how relevant this document excerpt is for answering the question. Reply with only a number from 0 to 10.",
                },
                {
                    "role": "user",
                    "content": f"Question: {query}\n\nExcerpt: {chunk['text']}",
                },
            ],
            temperature=0,
        )
        try:
            score = float(response.choices[0].message.content.strip())
        except ValueError:
            score = 0.0
        scored.append({**chunk, "rerank_score": score})

    scored.sort(key=lambda x: x["rerank_score"], reverse=True)
    return scored[:top_k]


def ask_with_reranking(question: str, retrieve_k: int = 10, final_k: int = 3):
    """RAG pipeline with reranking: retrieve more, then rerank to select the best."""
    # Retrieve a larger candidate set
    candidates = retrieve(question, top_k=retrieve_k)
    print(f"Retrieved {len(candidates)} candidates, reranking...\n")

    # Rerank
    top_chunks = rerank(question, candidates, top_k=final_k)

    print("Top chunks after reranking:")
    for c in top_chunks:
        print(f"  - {c['source']} | rerank score: {c['rerank_score']}")
    print()

    answer = generate_answer(question, top_chunks)
    print(f"Answer: {answer}")


ask_with_reranking("What is the sick leave policy?")

## Improvement 3 — Metadata Filtering

If you know which document the answer should come from, filter by source to avoid noise.

In [ ]:
def retrieve_filtered(query: str, source_file: str, top_k: int = 3) -> list[dict]:
    """Retrieve only from a specific source document."""
    query_embedding = get_embedding(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        where={"source": source_file},  # metadata filter
        include=["documents", "metadatas", "distances"],
    )

    return [
        {"text": text, "source": meta["source"], "score": 1 - dist}
        for text, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        )
    ]


# Only search within the product FAQ
context = retrieve_filtered("What is the refund policy?", source_file="product_faq.txt")
answer = generate_answer("What is the refund policy?", context)
print(answer)

## Evaluation — Is Your RAG Working?

A simple way to evaluate: run a set of known questions and check the answers.

In [ ]:
eval_set = [
    {
        "question": "How many days of annual leave do employees get?",
        "expected_keywords": ["10", "annual"],
    },
    {
        "question": "What is the refund policy for SmartAssist?",
        "expected_keywords": ["30-day", "30 day", "money-back"],
    },
    {
        "question": "How many WFH days are allowed per week?",
        "expected_keywords": ["2"],
    },
]


def simple_evaluate(eval_set: list[dict]):
    """Run evaluation questions and check if expected keywords appear in the answer."""
    results = []
    for item in eval_set:
        context = retrieve(item["question"])
        answer = generate_answer(item["question"], context)
        answer_lower = answer.lower()

        passed = any(kw.lower() in answer_lower for kw in item["expected_keywords"])
        results.append({"question": item["question"], "answer": answer, "passed": passed})

        status = "PASS" if passed else "FAIL"
        print(f"[{status}] {item['question']}")
        print(f"       {answer[:120]}...\n")

    passed_count = sum(1 for r in results if r["passed"])
    print(f"Result: {passed_count}/{len(results)} passed")


simple_evaluate(eval_set)

---
# Section 6 — Advanced: Agentic RAG

Instead of always retrieving, an **agent** can decide *when* and *what* to retrieve.

In [ ]:
# Define a "search" tool using the OpenAI tool-calling format
search_tool = {
    "type": "function",
    "function": {
        "name": "search_documents",
        "description": "Search the company knowledge base for information relevant to the user's question.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query to look up in the documents.",
                }
            },
            "required": ["query"],
        },
    },
}

AGENT_SYSTEM_PROMPT = "You are a helpful assistant with access to a company knowledge base. Use the search_documents tool when you need to look up information. If you can answer without searching, do so directly."


def agentic_ask(question: str) -> str:
    """
    Agentic RAG: the LLM decides whether and what to search.
    It can call search_documents, then use the results to answer.
    """
    import json

    messages = [
        {"role": "system", "content": AGENT_SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    # First call — the LLM may request a tool call
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=messages,
        tools=[search_tool],
        tool_choice="auto",
    )

    message = response.choices[0].message

    # If the LLM called a tool, execute it and send the result back
    if message.tool_calls:
        tool_call = message.tool_calls[0]
        args = json.loads(tool_call.function.arguments)
        search_query = args["query"]
        print(f"[Agent] Searching for: '{search_query}'")

        # Execute the search
        results = retrieve(search_query, top_k=3)
        context_text = "\n\n".join(f"Source: {r['source']}\n{r['text']}" for r in results)

        # Append the assistant's tool call and the tool result to the conversation
        messages.append(message)
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": context_text,
        })

        # Final call — LLM uses the search result to compose an answer
        final_response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=messages,
        )
        return final_response.choices[0].message.content

    # No tool call — LLM answered directly
    print("[Agent] Answering directly without search.")
    return message.content


# Test agentic RAG
print("--- Question that needs search ---")
answer = agentic_ask("What is our WFH policy?")
print(f"\nAnswer: {answer}")

In [ ]:
# Test: question that doesn't need search
print("--- Question that does NOT need search ---")
answer = agentic_ask("What does RAG stand for?")
print(f"\nAnswer: {answer}")

---
# Summary

You have built a complete RAG system that:

| Step | What it does |
|------|--------------|
| 1. Load Documents | Read files from disk |
| 2. Chunk | Split into retrievable pieces |
| 3. Embed | Convert text to vectors (1536-dim via `text-embedding-3-small`) |
| 4. Index | Store in ChromaDB vector database |
| 5. Retrieve | Find relevant chunks by semantic similarity |
| 6. Generate | LLM answers grounded in retrieved context |

### Improvements you also explored
- **Query rewriting** — clarify vague questions before retrieval
- **Reranking** — use LLM to select the best chunks from a larger candidate set
- **Metadata filtering** — restrict search to specific documents
- **Evaluation** — verify answers against known expected outputs
- **Agentic RAG** — let the LLM decide when and what to search

### Next Steps
- Add PDF loading with `pypdf`
- Persist the ChromaDB collection to disk (`chromadb.PersistentClient`)
- Try a dedicated reranker model (e.g. Cohere Rerank)
- Add conversation history for multi-turn chat
- Deploy as a FastAPI web service
- Swap to a different OpenRouter model — try `anthropic/claude-3-haiku`, `meta-llama/llama-3-8b-instruct`, etc.